# Práctica — Clase 13: Consumir APIs

En esta práctica vas a resolver 3 ejercicios consumiendo APIs públicas gratuitas que **no requieren autenticación**, convirtiendo cada respuesta en un DataFrame de pandas.

**Cómo trabajar cada ejercicio:**

1. Leé el enunciado y las pistas.
2. Completá las líneas marcadas con `# TODO: ...` en la celda de código. El resto del andamiaje ya está escrito, incluyendo un `try/except` y datos de ejemplo (`fallback`) por si no tenés internet en el momento de correrlo.
3. Corré la celda y comprobá el resultado.
4. Si te trabaste, desplegá la solución en el bloque `💡 Ver solución` que está debajo — pero intentalo primero por tu cuenta.

> Requiere `requests` y `pandas`. En Colab ya vienen instalados; en Jupyter local, si falta `requests`, corré `pip install requests` en una terminal (o descomentá el `!pip install requests` de la primera celda).

In [ ]:
# !pip install requests   # descomentar si hace falta en Jupyter local

import requests
import pandas as pd

## Ejercicio 1 — Chistes en un DataFrame (JSON plano)

**Enunciado:** Usá la [Official Joke API](https://official-joke-api.appspot.com/jokes/ten) (`https://official-joke-api.appspot.com/jokes/ten`) para traer 10 chistes al azar, convertirlos en un DataFrame y quedarte solo con las columnas útiles (`setup` y `punchline`), eliminando `type` e `id`.

**Pistas:**
- `requests.get(url, timeout=10)` te da un objeto respuesta; revisá `status_code` antes de confiar en los datos.
- `.json()` convierte el cuerpo de la respuesta en una lista de diccionarios de Python.
- Como el JSON es **plano** (ningún diccionario anidado dentro de otro), `pd.DataFrame(lista)` arma la tabla directo.
- `df.drop(columns=[...])` elimina columnas (usá `inplace=True` o reasigná el resultado).

In [ ]:
url_chistes = "https://official-joke-api.appspot.com/jokes/ten"

# Datos de respaldo por si no hay conexión al correr este notebook
chistes_fallback = [
    {"type": "general", "setup": "¿Cómo se dice pared en inglés?", "punchline": "Guall.", "id": 101},
    {"type": "general", "setup": "¿Qué hace una abeja en el gimnasio?", "punchline": "Zum-ba.", "id": 102},
    {"type": "programming", "setup": "¿Por qué el programador se quedó sin batería?",
     "punchline": "Porque olvidó cerrar el loop.", "id": 103},
]

# TODO 1: Hacé el GET a `url_chistes` (con timeout=10) dentro del try, guardando el resultado en `r`.
try:
    r = None  # <-- reemplazá esta línea por: r = requests.get(url_chistes, timeout=10)
except requests.exceptions.RequestException as e:
    r = None
    print("Sin conexión:", e)

# TODO 2: Si `r` no es None y `r.status_code == 200`, guardá `r.json()` en `chistes`.
#         Si no, usá `chistes_fallback`.
chistes = None  # <-- completar

# TODO 3: Armá un DataFrame llamado `df_chistes` a partir de `chistes`.
df_chistes = None  # <-- completar

# TODO 4: Eliminá las columnas "type" e "id" de `df_chistes`.


df_chistes

<details>
<summary>💡 Ver solución</summary>

```python
url_chistes = "https://official-joke-api.appspot.com/jokes/ten"

chistes_fallback = [
    {"type": "general", "setup": "¿Cómo se dice pared en inglés?", "punchline": "Guall.", "id": 101},
    {"type": "general", "setup": "¿Qué hace una abeja en el gimnasio?", "punchline": "Zum-ba.", "id": 102},
    {"type": "programming", "setup": "¿Por qué el programador se quedó sin batería?",
     "punchline": "Porque olvidó cerrar el loop.", "id": 103},
]

# 1. Hacemos el pedido protegido con try/except
try:
    r = requests.get(url_chistes, timeout=10)
except requests.exceptions.RequestException as e:
    r = None
    print("Sin conexión:", e)

# 2. Si el pedido funcionó (status 200), usamos los datos reales; si no, el fallback
if r is not None and r.status_code == 200:
    chistes = r.json()
else:
    chistes = chistes_fallback

# 3. El JSON es plano: pd.DataFrame() alcanza
df_chistes = pd.DataFrame(chistes)

# 4. Nos quedamos solo con las columnas útiles
df_chistes = df_chistes.drop(columns=["type", "id"])

df_chistes
```
</details>

## Ejercicio 2 — Calorías de una fruta (JSON anidado)

**Enunciado:** Usá la API de [Fruityvice](https://fruityvice.com/api/fruit/all) (`https://fruityvice.com/api/fruit/all`) para averiguar cuántas calorías tiene la banana. El JSON viene **anidado**: cada fruta trae un diccionario `nutritions` con sus propias claves (`calories`, `fat`, `sugar`...), así que vas a necesitar `pd.json_normalize()` en lugar de `pd.DataFrame()` directo.

**Pistas:**
- El flujo de pedido es el mismo de siempre: `requests.get()` → revisar `status_code` → `.json()`.
- `pd.json_normalize(lista)` aplana los diccionarios anidados: `nutritions["calories"]` se convierte en la columna `nutritions.calories`.
- Para filtrar una fila puntual: `df.loc[df["name"] == "Banana"]`.
- Para leer un valor de esa fila filtrada: `.iloc[0]["nutritions.calories"]`.

In [ ]:
url_frutas = "https://fruityvice.com/api/fruit/all"

# Datos de respaldo (misma estructura anidada que la API real) por si no hay conexión
frutas_fallback = [
    {"name": "Cherry", "id": 1, "family": "Rosaceae", "genus": "Prunus", "order": "Rosales",
     "nutritions": {"calories": 50, "fat": 1.0, "sugar": 8.0, "carbohydrates": 12.0, "protein": 1.0}},
    {"name": "Banana", "id": 2, "family": "Musaceae", "genus": "Musa", "order": "Zingiberales",
     "nutritions": {"calories": 96, "fat": 0.2, "sugar": 17.2, "carbohydrates": 22.0, "protein": 1.0}},
]

# TODO 1: Hacé el GET a `url_frutas` (con timeout=10) dentro del try.
try:
    r_frutas = None  # <-- reemplazá por: r_frutas = requests.get(url_frutas, timeout=10)
except requests.exceptions.RequestException as e:
    r_frutas = None
    print("Sin conexión:", e)

# TODO 2: Si `r_frutas` no es None y su status_code es 200, guardá `r_frutas.json()` en `frutas`.
#         Si no, usá `frutas_fallback`.
frutas = None  # <-- completar

# TODO 3: Convertí `frutas` en un DataFrame aplanado llamado `df_frutas` usando pd.json_normalize().
df_frutas = None  # <-- completar

# TODO 4: Filtrá la fila donde "name" == "Banana" y extraé el valor de "nutritions.calories".
calorias_banana = None  # <-- completar

print("Calorías de la banana:", calorias_banana)
df_frutas.head()

<details>
<summary>💡 Ver solución</summary>

```python
url_frutas = "https://fruityvice.com/api/fruit/all"

frutas_fallback = [
    {"name": "Cherry", "id": 1, "family": "Rosaceae", "genus": "Prunus", "order": "Rosales",
     "nutritions": {"calories": 50, "fat": 1.0, "sugar": 8.0, "carbohydrates": 12.0, "protein": 1.0}},
    {"name": "Banana", "id": 2, "family": "Musaceae", "genus": "Musa", "order": "Zingiberales",
     "nutritions": {"calories": 96, "fat": 0.2, "sugar": 17.2, "carbohydrates": 22.0, "protein": 1.0}},
]

# 1. Pedido protegido
try:
    r_frutas = requests.get(url_frutas, timeout=10)
except requests.exceptions.RequestException as e:
    r_frutas = None
    print("Sin conexión:", e)

# 2. Datos reales si el pedido salió bien, fallback si no
if r_frutas is not None and r_frutas.status_code == 200:
    frutas = r_frutas.json()
else:
    frutas = frutas_fallback

# 3. El JSON viene anidado (nutritions), así que aplanamos con json_normalize
df_frutas = pd.json_normalize(frutas)

# 4. Filtramos la fila de la banana y leemos su columna de calorías
banana = df_frutas.loc[df_frutas["name"] == "Banana"]
calorias_banana = banana.iloc[0]["nutritions.calories"]

print("Calorías de la banana:", calorias_banana)
df_frutas.head()
```
</details>

## Ejercicio 3 — Usuarios aleatorios con parámetros (JSON doblemente anidado)

**Enunciado:** Usá la API pública [RandomUser](https://randomuser.me/api/) (`https://randomuser.me/api/`) para generar 10 usuarios de prueba, pasando `results=10` como parámetro de la URL. Armá un DataFrame quedándote solo con el **nombre completo**, el **email** y el **país** de cada usuario.

**Pistas:**
- El número de usuarios se pide como parámetro: `requests.get(url, params={"results": 10}, timeout=10)`.
- La respuesta no es directamente una lista: es un diccionario con dos claves, `"results"` (la lista de usuarios) e `"info"`. A `pd.json_normalize()` le tenés que pasar `data["results"]`, no `data` entero.
- Cada usuario trae el nombre partido en `name.first` / `name.last` y el país en `location.country` — así quedan las columnas después de normalizar.
- Para armar el nombre completo podés combinar columnas: `df["name.first"] + " " + df["name.last"]`.

In [ ]:
url_usuarios = "https://randomuser.me/api/"

# Datos de respaldo (misma forma anidada que la API real: resultados dentro de "results")
usuarios_fallback = {
    "results": [
        {
            "gender": "female",
            "name": {"title": "Ms", "first": "Elena", "last": "Gomez"},
            "email": "elena.gomez@example.com",
            "location": {"country": "Spain"},
        },
        {
            "gender": "male",
            "name": {"title": "Mr", "first": "Diego", "last": "Ramirez"},
            "email": "diego.ramirez@example.com",
            "location": {"country": "Argentina"},
        },
        {
            "gender": "female",
            "name": {"title": "Mrs", "first": "Aiko", "last": "Tanaka"},
            "email": "aiko.tanaka@example.com",
            "location": {"country": "Japan"},
        },
    ]
}

# TODO 1: Hacé el GET a `url_usuarios` pasando params={"results": 10} y timeout=10.
try:
    r_usuarios = None  # <-- reemplazá por: r_usuarios = requests.get(url_usuarios, params={"results": 10}, timeout=10)
except requests.exceptions.RequestException as e:
    r_usuarios = None
    print("Sin conexión:", e)

# TODO 2: Si `r_usuarios` no es None y su status_code es 200, guardá `r_usuarios.json()` en `data`.
#         Si no, usá `usuarios_fallback`.
data = None  # <-- completar

# TODO 3: Normalizá la lista que está en `data["results"]` con pd.json_normalize() en `df_usuarios`.
df_usuarios = None  # <-- completar

# TODO 4: Creá una columna "nombre_completo" combinando "name.first" y "name.last".

# TODO 5: Quedate solo con las columnas "nombre_completo", "email" y "location.country".
df_final = None  # <-- completar

df_final

<details>
<summary>💡 Ver solución</summary>

```python
url_usuarios = "https://randomuser.me/api/"

usuarios_fallback = {
    "results": [
        {
            "gender": "female",
            "name": {"title": "Ms", "first": "Elena", "last": "Gomez"},
            "email": "elena.gomez@example.com",
            "location": {"country": "Spain"},
        },
        {
            "gender": "male",
            "name": {"title": "Mr", "first": "Diego", "last": "Ramirez"},
            "email": "diego.ramirez@example.com",
            "location": {"country": "Argentina"},
        },
        {
            "gender": "female",
            "name": {"title": "Mrs", "first": "Aiko", "last": "Tanaka"},
            "email": "aiko.tanaka@example.com",
            "location": {"country": "Japan"},
        },
    ]
}

# 1. Pedido con parámetros (results=10), protegido
try:
    r_usuarios = requests.get(url_usuarios, params={"results": 10}, timeout=10)
except requests.exceptions.RequestException as e:
    r_usuarios = None
    print("Sin conexión:", e)

# 2. Datos reales si salió bien, fallback si no
if r_usuarios is not None and r_usuarios.status_code == 200:
    data = r_usuarios.json()
else:
    data = usuarios_fallback

# 3. La lista de usuarios está anidada dentro de la clave "results"
df_usuarios = pd.json_normalize(data["results"])

# 4. Combinamos nombre y apellido en una sola columna
df_usuarios["nombre_completo"] = df_usuarios["name.first"] + " " + df_usuarios["name.last"]

# 5. Nos quedamos solo con las columnas que nos interesan
df_final = df_usuarios[["nombre_completo", "email", "location.country"]]

df_final
```
</details>